In [1]:
from itertools import combinations
import math
import random
import time

##  Brute force (exhaustive search)
Try every way of choosing k − 1 cut positions out of the gaps 1 … n − 1, and keep the split whose heaviest group is smallest.

In [2]:
def sums(w):
    """Return P where P[0] = 0 and P[i] = w[0] + ... + w[i-1].  len(P) == len(w) + 1."""
    prefix = [0]
    for weight in w:
        prefix.append(prefix[-1] + weight)
    return prefix


def bruteForce(w, k):
    n = len(w)
    if not 1 <= k <= n:
        raise ValueError("k must be between 1 and len(w)")

    P = sums(w)
    best = math.inf
    bestCuts = None

    for chosen in combinations(range(1, n), k - 1):
        cuts = [0] + list(chosen) + [n]
        largest = max(P[cuts[j]] - P[cuts[j - 1]] for j in range(1, k + 1))
        if largest < best:
            best = largest
            bestCuts = cuts

    groups = [w[bestCuts[j - 1]:bestCuts[j]] for j in range(1, k + 1)]
    return best, groups

In [3]:
handTests = [
    # (w, k, expected answer, expected groups or None if several splits tie)
    ([7, 2, 5, 10, 8], 2, 18, [[7, 2, 5], [10, 8]]),
    ([7, 2, 5, 10, 8], 3, 14, [[7, 2, 5], [10], [8]]),
    ([4, 3, 6, 1], 2, 7, [[4, 3], [6, 1]]),
]
for w, k, wantAns, wantGroups in handTests:
    ans, groups = bruteForce(w, k)
    ok = ans == wantAns and (wantGroups is None or groups == wantGroups)
    print("PASS" if ok else "FAIL", w, "k =", k, "->", ans, groups)

PASS [7, 2, 5, 10, 8] k = 2 -> 18 [[7, 2, 5], [10, 8]]
PASS [7, 2, 5, 10, 8] k = 3 -> 14 [[7, 2, 5], [10], [8]]
PASS [4, 3, 6, 1] k = 2 -> 7 [[4, 3], [6, 1]]


## Answer checker
It returns `(True, "ok")` if the output is valid, or `(False, reason)` for the first rule broken.

Rules: exactly k groups; no empty group; the boxes are all there, once each, in the original order; the heaviest group equals the reported answer.



In [4]:
def checkSplit(w, k, answer, groups):
    if len(groups) != k:
        return False, f"expected {k} groups, got {len(groups)}"
    if any(len(group) == 0 for group in groups):
        return False, "empty group"
    if [weight for group in groups for weight in group] != w:
        return False, "boxes missing, repeated or out of order"
    heaviest = max(sum(group) for group in groups)
    if heaviest != answer:
        return False, f"heaviest group is {heaviest}, but reported {answer}"
    return True, "ok"

In [5]:
w = [7, 2, 5, 10, 8]
checkerTests = [
    # (k, answer, groups, should it pass?)
    (2, 18, [[7, 2, 5], [10, 8]], True),
    (2, 18, [[7, 2, 5], [10, 8, 8]], False),     # 8 used twice
    (2, 18, [[7, 2, 5], [8, 10]], False),        # right number, order broken
    (2, 32, [[7, 2, 5, 10, 8]], False),          # only 1 group
    (3, 18, [[7, 2, 5], [], [10, 8]], False),    # empty group
    (2, 14, [[7, 2, 5], [10, 8]], False),        # reported number is wrong
]
for k, answer, groups, shouldPass in checkerTests:
    ok, reason = checkSplit(w, k, answer, groups)
    print("PASS" if ok == shouldPass else "FAIL", groups, "->", ok, reason)

PASS [[7, 2, 5], [10, 8]] -> True ok
PASS [[7, 2, 5], [10, 8, 8]] -> False boxes missing, repeated or out of order
PASS [[7, 2, 5], [8, 10]] -> False boxes missing, repeated or out of order
PASS [[7, 2, 5, 10, 8]] -> False expected 2 groups, got 1
PASS [[7, 2, 5], [], [10, 8]] -> False empty group
PASS [[7, 2, 5], [10, 8]] -> False heaviest group is 18, but reported 14


## 3. Edge cases 
Work out each expected answer on paper first, then type it in place of `None`.

In [ ]:
edgeCases = [
    # (name, w, k, expected answer)
    ("one task",                 [5],                1, 5),
    ("k = 1 (answer = total)",   [7, 2, 5, 10, 8],   1, 32),
    ("k = n (answer = biggest)", [7, 2, 5, 10, 8],   5, 10),
    ("all weights equal",        [4, 4, 4, 4, 4, 4], 3, 8),
    ("one huge weight, k = 2",   [1, 1, 100, 1, 1],  2, 102),
    ("one huge weight, k = 3",   [1, 1, 100, 1, 1],  3, 100),
    ("sorted input",             [1, 2, 3, 4, 5],    2, 9),
    ("packing gives too few",    [1, 1, 1, 10],      3, 10),
]
for name, w, k, want in edgeCases:
    ans, groups = bruteForce(w, k)
    ok, reason = checkSplit(w, k, ans, groups)
    print("PASS" if (ans == want and ok) else "FAIL", name, "->", ans, groups, reason)

# Bad input must raise ValueError, not crash or return nonsense
for w, k in [([7, 2, 5], 0), ([7, 2, 5], 4)]:
    try:
        bruteForce(w, k)
        print("FAIL: no error for k =", k)
    except ValueError:
        print("PASS: ValueError for k =", k)

PASS one task -> 5 [[5]] ok
PASS k = 1 (answer = total) -> 32 [[7, 2, 5, 10, 8]] ok
PASS k = n (answer = biggest) -> 10 [[7], [2], [5], [10], [8]] ok
PASS all weights equal -> 8 [[4, 4], [4, 4], [4, 4]] ok
FAIL one huge weight, k = 2 -> 102 [[1, 1], [100, 1, 1]] ok
PASS one huge weight, k = 3 -> 100 [[1, 1], [100], [1, 1]] ok
PASS sorted input -> 9 [[1, 2, 3], [4, 5]] ok
PASS packing gives too few -> 10 [[1], [1, 1], [10]] ok
PASS: ValueError for k = 0
PASS: ValueError for k = 4


##  Random checks
This only tests brute force against the checker and the bounds max(w) ≤ answer ≤ sum(w).

In [7]:
def randomCase(rng, maxN=12, maxW=20):
    n = rng.randint(1, maxN)
    k = rng.randint(1, n)
    weights = [rng.randint(1, maxW) for index in range(n)]
    return weights, k


def runRandomChecks(methods, trials=1000, seed=456):
    rng = random.Random(seed)          # fixed seed, so anyone can repeat the exact same run
    for t in range(trials):
        w, k = randomCase(rng)
        refAns, refGroups = bruteForce(w, k)

        ok, reason = checkSplit(w, k, refAns, refGroups)
        if not ok:
            print("FAIL: brute force", t, w, k, reason)
            return False
        if not max(w) <= refAns <= sum(w):
            print("FAIL: bounds", t, w, k, refAns)
            return False

        for name, method in methods.items():
            ans, groups = method(w, k)
            ok, reason = checkSplit(w, k, ans, groups)
            if ans != refAns or not ok:
                print("FAIL:", name, t, w, k, ans, groups, reason)
                return False

    print("All", trials, "random checks passed for: brute force", *methods)
    return True


methods = {}   # on Friday: {"DP": dpPartition, "binary search": binarySearchPartition}
runRandomChecks(methods)

All 1000 random checks passed for: brute force


True

## Stopping point 
Time brute force with k = n/2 for n = 10, 12, 14, … and stop after the first run that takes over 60 seconds. Each +2 tasks makes it about 4 times slower, so you can predict the next time before you run it.


In [8]:
def timeOnce(n, seed=0):
    rng = random.Random(seed)
    w = [rng.randint(1, 1000) for index in range(n)]
    k = n // 2
    start = time.perf_counter()
    bruteForce(w, k)
    return time.perf_counter() - start


for n in range(10, 100, 2):
    k = n // 2
    seconds = timeOnce(n)
    splitCount = math.comb(n - 1, k - 1)
    print(n, k, splitCount, f"{seconds:.2f} seconds")
    if seconds > 60:
        break

10 5 126 0.00 seconds
12 6 462 0.00 seconds
14 7 1716 0.01 seconds
16 8 6435 0.06 seconds
18 9 24310 0.39 seconds
20 10 92378 1.21 seconds
22 11 352716 4.41 seconds
24 12 1352078 13.29 seconds
26 13 5200300 33.23 seconds
28 14 20058300 122.40 seconds
